## Company Policy RAG Assistant – Implementation Summary

The Company Policy RAG Assistant is a Python-based prototype designed to help employees quickly query and retrieve information from company HR policies. HR documents (e.g., Leave Policy, Code of Conduct, Health Insurance Policy) are converted to PDFs and ingested into a vector database. Text chunks from these documents are embedded using OpenAI embeddings, enabling semantic search.

When a user asks a question, the system retrieves the most relevant policy sections and passes them to a large language model (e.g., GPT-4o-mini) to generate concise, context-aware answers. Implemented on Google Colab, this pipeline demonstrates a complete RAG workflow—from document ingestion and embedding to retrieval and LLM-based answer generation—allowing employees to access policy information efficiently without reading full documents.


Run all cells in order. Ensure you are in the notebook's directory (so `knowledge_base/` and this notebook are in the same folder) and that `OPENAI_API_KEY` is set in `.env`.

In [ ]:
%pip install numpy langchain langchain_community langchain_text_splitters gradio chromadb sentence-transformers pypdf openai tiktoken

In [204]:
%pip install --upgrade chromadb langchain_chroma

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 59.9 kB/s eta 0:00:0000:0200:12
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.0.4
    Uninstalling langchain-core-1.0.4:
      Successfully uninstalled langchain-core-1.0.4
  Attempting uninstall: chromadb
    Found existing installation: chromadb 1.3.4
    Uninstalling chromadb-1.3.4:
      Successfully uninstalled chromadb-1.3.4
  Attempting uninstall: langchain_chroma
    Found existing installation: langchain-chroma 1.0.0
    Uninstalling langchain-chroma-1.0.0:
      Successfully uninstalled langchain-chroma-1.0.0

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [205]:
%pip install pypandoc numpy

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [215]:
# Imports and paths
import os
import random
import sqlite3
import shutil
from pathlib import Path

from dotenv import load_dotenv
import pandas as pd
import gradio as gr

from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
try:
    from google.colab import files  # type: ignore  # only available in Colab
except ImportError:
    files = None  # Not in Colab; upload cell will be skipped when running locally

load_dotenv(override=True)

BASE_DIR = Path.cwd()
KB_DIR = BASE_DIR / "knowledge_base"
CHROMA_DIR = BASE_DIR / "vector_db"

print("KB_DIR:", KB_DIR)
print("CHROMA_DIR:", CHROMA_DIR)
# print("DB_PATH:", DB_PATH)

KB_DIR: /Users/olugbengafalodun/projects/llm_engineering/week5/community-contributions/falodun.solomon/knowledge_base
CHROMA_DIR: /Users/olugbengafalodun/projects/llm_engineering/week5/community-contributions/falodun.solomon/vector_db


In [216]:
os.makedirs("knowledge_base", exist_ok=True)
os.makedirs("vector_db", exist_ok=True)

In [217]:
# Upload Your HR Policy Documents (Colab only; locally, place files in knowledge_base/)
if files is not None:
    uploaded = files.upload()
else:
    uploaded = {}
    print("Running locally: place HR policy documents in the knowledge_base/ folder.")

for file in uploaded.keys():
    shutil.move(file, f"knowledge_base/{file}")

Running locally: place HR policy documents in the knowledge_base/ folder.


In [218]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader

KB_DIR = Path("knowledge_base")

documents = []

for file in KB_DIR.glob("*.pdf"):
    loader = PyPDFLoader(str(file))
    documents.extend(loader.load())

for file in KB_DIR.glob("*.md"):
    loader = TextLoader(str(file))
    documents.extend(loader.load())

print(f"Loaded {len(documents)} document pages")

Loaded 10 document pages


In [219]:
# Split Documents Into Chunks

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")

Created 18 chunks


In [220]:
# Use OpenAI embeddings (avoids sentence_transformers/numpy issues). Requires OPENAI_API_KEY in .env.
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [221]:
# Store embeddings in ChromaDB
from langchain_chroma import Chroma

if not chunks:
    raise ValueError(
        "No documents to embed. Add PDF or .md files to knowledge_base/ and re-run "
        "the document-loading and splitter cells above."
    )

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="vector_db",
    collection_name="hr_policies" 
)

if hasattr(vectordb, "persist"):
    vectordb.persist()

print("Vector database created")


InternalError: Database error: error returned from database: (code: 1) no such table: tenants

In [212]:
# Create a Retriever - This will fetch relevant policy sections
retriever = vectordb.as_retriever(
    search_kwargs={"k":3}
)

In [ ]:
question = "How many maternity leave weeks do employees get?"

docs = retriever.invoke(question)

for doc in docs:
    print(doc.page_content)
    print("----")

In [ ]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging
# You can choose whichever providers you like - or all Ollama

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openrouter_base_url = os.getenv('OPENROUTER_BASE_URL')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

In [ ]:

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

template = """Answer based only on the following context:
{context}

Question: {question}
Answer:"""
prompt = ChatPromptTemplate.from_template(template)

retrieval_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


def chat_hr_policy(message, history):
    """Gradio chat handler: run RAG and return (answer, context_md) for ChatInterface + additional_outputs."""
    if not message or not message.strip():
        return "", ""
    docs = retriever.invoke(message)
    context_md = "\n\n---\n\n".join(
        f"**Excerpt {i+1}:**\n{doc.page_content}" for i, doc in enumerate(docs)
    )
    answer = retrieval_chain.invoke(message)
    return answer, context_md


# Optional: test the chain in the notebook
# response = retrieval_chain.invoke("What is the maternity leave policy?")
# print(response)

In [ ]:
# Gradio: chat + second output (retrieved context with scores) in a toggleable Accordion
with gr.Blocks(title="Employee HR Policy Assistant") as demo:
    gr.Markdown("# Employee HR Policy Assistant")
    gr.Markdown("Ask for HR policy information, you will get the relevant policy section")
    with gr.Accordion("Retrieved context (toggle to view)", open=False):
        context_out = gr.Markdown(value="", label="Context with scores")
    chat_if = gr.ChatInterface(
        fn=chat_hr_policy,
        additional_outputs=[context_out],
    )
demo.launch(inbrowser=True, theme=gr.themes.Soft())